# REHAB24-6 Preprocessing — Full Repetitions to Fixed-Length F32 + Velocity + Bones + Angles

This notebook converts the **full-repetition pose clips** extracted from REHAB24-6 into a **training-ready fixed-length dataset**.
## What this notebook does
1. Loads full-repetition pose clips saved as `(T, 33, 4)` with:
   - `x, y, z, visibility`
2. Uniformly resamples each repetition to **32 frames**
3. Adds **velocity features**:
   - `vx, vy, vz`
4. Adds **bone features**:
   - `bx, by, bz`
5. Extracts **8 key joint-angle features** per frame
6. Saves training-ready artifacts

## Output files
- `X_pose.npy` → shape `(N, 32, 33, 10)`
- `X_angles.npy` → shape `(N, 32, 8)`
- `y.npy` → shape `(N,)`
- `index_clean.csv`
- `dataset_summary.json`


In [1]:
# 1) Imports
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

## 2) Paths and settings

In [2]:
# 2) Paths and settings

# Input from the full-repetition extraction notebook
POSE_DIR = Path("pose_clips/full_reps/camera17")
POSE_INDEX_CSV = POSE_DIR / "pose_index_full_reps.csv"

# Output for training-ready data
OUT_DIR = Path("artifacts/artifacts_rehab24_fullreps_F32")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed sequence length for training
TARGET_FRAMES = 32

# Save dtype
SAVE_DTYPE = np.float32

print("POSE_DIR:", POSE_DIR.resolve())
print("POSE_INDEX_CSV:", POSE_INDEX_CSV.resolve())
print("OUT_DIR:", OUT_DIR.resolve())
print("POSE_DIR exists?", POSE_DIR.exists())
print("POSE_INDEX_CSV exists?", POSE_INDEX_CSV.exists())

POSE_DIR: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\pose_clips\full_reps\camera17
POSE_INDEX_CSV: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\pose_clips\full_reps\camera17\pose_index_full_reps.csv
OUT_DIR: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\artifacts\artifacts_rehab24_fullreps_F32
POSE_DIR exists? True
POSE_INDEX_CSV exists? True


## 3) Load pose index

In [3]:
# 3) Load pose index
pose_index = pd.read_csv(POSE_INDEX_CSV)

print("pose_index shape:", pose_index.shape)
print("Columns:", pose_index.columns.tolist())
pose_index.head()

pose_index shape: (1072, 17)
Columns: ['video_id', 'repetition_number', 'exercise_id', 'person_id', 'first_frame', 'last_frame', 'cam17_orientation', 'mocap_erroneous', 'exercise_subtype', 'lights_on', 'extra_person_in_cam17', 'extra_person_in_cam18', 'correctness', 'video_path_abs', 'pose_path', 'num_pose_frames', 'pose_shape']


,video_id,repetition_number,exercise_id,person_id,first_frame,last_frame,cam17_orientation,mocap_erroneous,exercise_subtype,lights_on,extra_person_in_cam17,extra_person_in_cam18,correctness,video_path_abs,pose_path,num_pose_frames,pose_shape
0,PM_000,1,1,1,180,377,front,0,right arm,0,3,0,1,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\d...,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\c...,198,"(198, 33, 4)"
1,PM_000,2,1,1,378,620,front,0,right arm,0,3,0,1,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\d...,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\c...,243,"(243, 33, 4)"
2,PM_000,3,1,1,621,865,front,0,right arm,0,3,0,1,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\d...,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\c...,245,"(245, 33, 4)"
3,PM_000,4,1,1,866,1085,front,0,right arm,0,3,3,1,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\d...,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\c...,220,"(220, 33, 4)"
4,PM_000,5,1,1,1086,1265,front,0,right arm,0,3,3,1,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\d...,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\c...,180,"(180, 33, 4)"


In [4]:
# 3.1) Required columns check
required_cols = [
    "video_id", "repetition_number", "exercise_id", "person_id",
    "first_frame", "last_frame", "correctness", "pose_path"
]

missing_cols = [c for c in required_cols if c not in pose_index.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in pose_index_full_reps.csv: {missing_cols}")

pose_index = pose_index.copy()
pose_index["exercise_id"] = pose_index["exercise_id"].astype(int)
pose_index["person_id"] = pose_index["person_id"].astype(int)
pose_index["repetition_number"] = pose_index["repetition_number"].astype(int)
pose_index["first_frame"] = pose_index["first_frame"].astype(int)
pose_index["last_frame"] = pose_index["last_frame"].astype(int)
pose_index["correctness"] = pose_index["correctness"].astype(int)

print("Unique exercises:", sorted(pose_index["exercise_id"].unique().tolist()))
print("Unique persons:", sorted(pose_index["person_id"].unique().tolist()))
print("Labels:", pose_index["correctness"].value_counts().sort_index().to_dict())

Unique exercises: [1, 2, 3, 4, 5, 6]
Unique persons: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Labels: {0: 504, 1: 568}


## 4) Preprocessing utilities

This section defines:
- uniform temporal resampling
- velocity features
- bone features
- angle features
- quick sanity checks


In [5]:
# 4) Preprocessing utilities

def resample_pose_clip_uniform(clip: np.ndarray, target_frames: int = 32) -> np.ndarray:
    """
    Uniformly resample a pose clip from (T, 33, 4) to (target_frames, 33, 4).
    """
    clip = np.asarray(clip, dtype=np.float32)

    if clip.ndim != 3 or clip.shape[1] != 33 or clip.shape[2] != 4:
        raise ValueError(f"Expected clip shape (T,33,4), got {clip.shape}")

    T = clip.shape[0]
    if T == 0:
        return np.zeros((target_frames, 33, 4), dtype=np.float32)

    if T == target_frames:
        return clip.astype(np.float32, copy=False)

    idxs = np.linspace(0, T - 1, target_frames)
    idxs = np.round(idxs).astype(int)
    idxs = np.clip(idxs, 0, T - 1)
    return clip[idxs].astype(np.float32, copy=False)


def add_velocity_features(pose_seq: np.ndarray) -> np.ndarray:
    """
    Convert (T,33,4) -> (T,33,7)
    Adds velocity features vx, vy, vz.
    Output channel order:
    [x, y, z, visibility, vx, vy, vz]
    """
    pose_seq = np.asarray(pose_seq, dtype=np.float32)

    if pose_seq.ndim != 3 or pose_seq.shape[1] != 33 or pose_seq.shape[2] != 4:
        raise ValueError(f"Expected pose_seq shape (T,33,4), got {pose_seq.shape}")

    xyz = pose_seq[..., :3]
    vis = pose_seq[..., 3:4]

    vel = np.zeros_like(xyz, dtype=np.float32)
    vel[1:] = xyz[1:] - xyz[:-1]

    return np.concatenate([xyz, vis, vel], axis=-1).astype(np.float32, copy=False)


def add_bone_features(pose_seq: np.ndarray) -> np.ndarray:
    """
    Convert (T,33,7) -> (T,33,10)
    Adds bone vectors bx, by, bz using a useful subset of MediaPipe connections.
    Output channel order:
    [x, y, z, visibility, vx, vy, vz, bx, by, bz]
    """
    pose_seq = np.asarray(pose_seq, dtype=np.float32)

    if pose_seq.ndim != 3 or pose_seq.shape[1] != 33 or pose_seq.shape[2] != 7:
        raise ValueError(f"Expected pose_seq shape (T,33,7), got {pose_seq.shape}")

    xyz = pose_seq[..., :3]
    bones = np.zeros_like(xyz, dtype=np.float32)

    connections = [
        (11, 13), (13, 15),   # left arm
        (12, 14), (14, 16),   # right arm
        (23, 25), (25, 27),   # left leg
        (24, 26), (26, 28),   # right leg
        (11, 12),             # shoulders
        (23, 24),             # hips
        (11, 23), (12, 24)    # torso
    ]

    for parent, child in connections:
        bones[:, child] = xyz[:, child] - xyz[:, parent]

    return np.concatenate([pose_seq, bones], axis=-1).astype(np.float32, copy=False)


def angle_3pts(a: np.ndarray, b: np.ndarray, c: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """
    Compute the angle at point b formed by points a-b-c.
    Inputs have shape (..., 3), output shape (...,).
    """
    ba = a - b
    bc = c - b

    ba_norm = np.linalg.norm(ba, axis=-1, keepdims=True)
    bc_norm = np.linalg.norm(bc, axis=-1, keepdims=True)

    ba = ba / np.maximum(ba_norm, eps)
    bc = bc / np.maximum(bc_norm, eps)

    cosang = np.sum(ba * bc, axis=-1)
    cosang = np.clip(cosang, -1.0, 1.0)
    return np.arccos(cosang).astype(np.float32)


def extract_angle_features(pose_seq: np.ndarray) -> np.ndarray:
    """
    pose_seq: (T, 33, C), expects xyz in [:, :, :3]
    returns: (T, 8) angle features in radians
    Angle order:
    [left_elbow, right_elbow, left_knee, right_knee,
     left_shoulder, right_shoulder, left_hip, right_hip]
    """
    pose_seq = np.asarray(pose_seq, dtype=np.float32)

    if pose_seq.ndim != 3 or pose_seq.shape[1] != 33 or pose_seq.shape[2] < 3:
        raise ValueError(f"Expected pose_seq shape (T,33,C>=3), got {pose_seq.shape}")

    xyz = pose_seq[..., :3]

    L_SHOULDER, R_SHOULDER = 11, 12
    L_ELBOW,    R_ELBOW    = 13, 14
    L_WRIST,    R_WRIST    = 15, 16
    L_HIP,      R_HIP      = 23, 24
    L_KNEE,     R_KNEE     = 25, 26
    L_ANKLE,    R_ANKLE    = 27, 28

    left_elbow   = angle_3pts(xyz[:, L_SHOULDER], xyz[:, L_ELBOW], xyz[:, L_WRIST])
    right_elbow  = angle_3pts(xyz[:, R_SHOULDER], xyz[:, R_ELBOW], xyz[:, R_WRIST])
    left_knee    = angle_3pts(xyz[:, L_HIP], xyz[:, L_KNEE], xyz[:, L_ANKLE])
    right_knee   = angle_3pts(xyz[:, R_HIP], xyz[:, R_KNEE], xyz[:, R_ANKLE])
    left_shoulder  = angle_3pts(xyz[:, L_ELBOW], xyz[:, L_SHOULDER], xyz[:, L_HIP])
    right_shoulder = angle_3pts(xyz[:, R_ELBOW], xyz[:, R_SHOULDER], xyz[:, R_HIP])
    left_hip     = angle_3pts(xyz[:, L_SHOULDER], xyz[:, L_HIP], xyz[:, L_KNEE])
    right_hip    = angle_3pts(xyz[:, R_SHOULDER], xyz[:, R_HIP], xyz[:, R_KNEE])

    angles = np.stack([
        left_elbow, right_elbow,
        left_knee, right_knee,
        left_shoulder, right_shoulder,
        left_hip, right_hip
    ], axis=-1).astype(np.float32)

    return angles


def quick_pose_checks(arr: np.ndarray) -> dict:
    arr = np.asarray(arr)

    xyz = arr[..., :3]
    vis = arr[..., 3] if arr.shape[-1] > 3 else np.array([], dtype=np.float32)

    out = {
        "shape": tuple(arr.shape),
        "nan_ratio_xyz": float(np.isnan(xyz).sum() / xyz.size) if xyz.size else np.nan,
    }

    if vis.size:
        out["min_vis"] = float(np.nanmin(vis))
        out["max_vis"] = float(np.nanmax(vis))
    else:
        out["min_vis"] = np.nan
        out["max_vis"] = np.nan

    return out

## 5) Preview one raw clip and one processed clip

In [6]:
# 5) Preview one raw clip and one processed clip
loc = 0
sample_path = Path(pose_index.loc[loc, "pose_path"])
sample_clip = np.load(sample_path)

print("Sample path:", sample_path)
print("Raw sample shape:", sample_clip.shape)
print("Raw sample checks:", quick_pose_checks(sample_clip))

sample_resampled = resample_pose_clip_uniform(sample_clip, TARGET_FRAMES)
sample_plus_vel = add_velocity_features(sample_resampled)
sample_plus_all = add_bone_features(sample_plus_vel)
sample_angles = extract_angle_features(sample_plus_all)

print("Resampled sample shape:", sample_resampled.shape)
print("After velocity shape:", sample_plus_vel.shape)
print("After bones shape:", sample_plus_all.shape)
print("Angle feature shape:", sample_angles.shape)
print("Processed sample checks:", quick_pose_checks(sample_plus_all))

Sample path: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\pose_clips\full_reps\camera17\PM_000_r1_f180-377_pose_full.npy
Raw sample shape: (198, 33, 4)
Raw sample checks: {'shape': (198, 33, 4), 'nan_ratio_xyz': 0.0, 'min_vis': 0.7694001197814941, 'max_vis': 0.9999977350234985}
Resampled sample shape: (32, 33, 4)
After velocity shape: (32, 33, 7)
After bones shape: (32, 33, 10)
Angle feature shape: (32, 8)
Processed sample checks: {'shape': (32, 33, 10), 'nan_ratio_xyz': 0.0, 'min_vis': 0.7707237601280212, 'max_vis': 0.9999977350234985}


## 6) Build the fixed-length dataset

This creates:
- `X_pose` with joints + visibility + velocity + bones
- `X_angles` with 8 key angles per frame
- `y` from `correctness`
- `index_clean` with metadata


In [8]:
# 6) Build the fixed-length dataset

X_list = []
A_list = []
rows = []
failed = 0

for _, row in tqdm(pose_index.iterrows(), total=len(pose_index), desc="Preprocessing pose clips"):
    pose_path = Path(row["pose_path"])

    try:
        clip = np.load(pose_path)

        if clip.ndim != 3 or clip.shape[1:] != (33, 4):
            raise ValueError(f"Unexpected clip shape: {clip.shape}")

        clip_fixed = resample_pose_clip_uniform(clip, TARGET_FRAMES)
        clip_fixed = add_velocity_features(clip_fixed)
        clip_fixed = add_bone_features(clip_fixed)

        angle_feat = extract_angle_features(clip_fixed)

        clip_fixed = np.nan_to_num(
            clip_fixed, nan=0.0, posinf=0.0, neginf=0.0
        ).astype(SAVE_DTYPE, copy=False)

        angle_feat = np.nan_to_num(
            angle_feat, nan=0.0, posinf=0.0, neginf=0.0
        ).astype(SAVE_DTYPE, copy=False)

        X_list.append(clip_fixed)
        A_list.append(angle_feat)

        r = row.to_dict()
        r["orig_num_pose_frames"] = int(clip.shape[0])
        r["fixed_num_pose_frames"] = int(TARGET_FRAMES)
        rows.append(r)

    except Exception as e:
        failed += 1
        print(f"[FAILED] {pose_path.name}: {e}")

if not X_list:
    raise RuntimeError("No pose clips were successfully processed.")

X_pose = np.stack(X_list).astype(SAVE_DTYPE, copy=False)
X_angles = np.stack(A_list).astype(SAVE_DTYPE, copy=False)
index_clean = pd.DataFrame(rows)
y = index_clean["correctness"].to_numpy(dtype=np.int64)

print("X_pose shape:", X_pose.shape)
print("X_angles shape:", X_angles.shape)
print("y shape:", y.shape)
print("index_clean shape:", index_clean.shape)
print("Failed:", failed)

print("NaNs in X_pose:", np.isnan(X_pose).sum())
print("NaNs in X_angles:", np.isnan(X_angles).sum())

Preprocessing pose clips: 100%|██████████| 1072/1072 [00:01<00:00, 934.23it/s] 

X_pose shape: (1072, 32, 33, 10)
X_angles shape: (1072, 32, 8)
y shape: (1072,)
index_clean shape: (1072, 19)
Failed: 0
NaNs in X_pose: 0
NaNs in X_angles: 0


## 7) Save outputs

In [9]:
# 7) Save outputs

X_PATH = OUT_DIR / "X_pose.npy"
A_PATH = OUT_DIR / "X_angles.npy"
Y_PATH = OUT_DIR / "y.npy"
INDEX_PATH = OUT_DIR / "index_clean.csv"
SUMMARY_PATH = OUT_DIR / "dataset_summary.json"

np.save(X_PATH, X_pose.astype(SAVE_DTYPE, copy=False))
np.save(A_PATH, X_angles.astype(SAVE_DTYPE, copy=False))
np.save(Y_PATH, y.astype(np.int64, copy=False))
index_clean.to_csv(INDEX_PATH, index=False)

summary = {
    "source_pose_dir": str(POSE_DIR.resolve()),
    "source_pose_index_csv": str(POSE_INDEX_CSV.resolve()),
    "num_samples": int(len(index_clean)),
    "target_frames": int(TARGET_FRAMES),
    "shape_X_pose": list(X_pose.shape),
    "shape_X_angles": list(X_angles.shape),
    "shape_y": list(y.shape),
    "dtype_X_pose": str(X_pose.dtype),
    "dtype_X_angles": str(X_angles.dtype),
    "labels": {str(k): int(v) for k, v in index_clean["correctness"].value_counts().sort_index().to_dict().items()},
    "num_exercises": int(index_clean["exercise_id"].nunique()),
    "num_persons": int(index_clean["person_id"].nunique()),
    "min_orig_frames": int(index_clean["orig_num_pose_frames"].min()),
    "mean_orig_frames": float(index_clean["orig_num_pose_frames"].mean()),
    "max_orig_frames": int(index_clean["orig_num_pose_frames"].max()),
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", X_PATH.resolve())
print("Saved:", A_PATH.resolve())
print("Saved:", Y_PATH.resolve())
print("Saved:", INDEX_PATH.resolve())
print("Saved:", SUMMARY_PATH.resolve())

Saved: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\artifacts\artifacts_rehab24_fullreps_F32\X_pose.npy
Saved: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\artifacts\artifacts_rehab24_fullreps_F32\X_angles.npy
Saved: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\artifacts\artifacts_rehab24_fullreps_F32\y.npy
Saved: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\artifacts\artifacts_rehab24_fullreps_F32\index_clean.csv
Saved: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\artifacts\artifacts_rehab24_fullreps_F32\dataset_summary.json


## 8) Sanity checks

In [10]:
# 8) Sanity checks

print("Label counts:")
print(index_clean["correctness"].value_counts().sort_index())

print("\nExercise counts:")
print(index_clean["exercise_id"].value_counts().sort_index())

print("\nUnique persons:", index_clean["person_id"].nunique())
print("Min / mean / max original frames:",
      int(index_clean["orig_num_pose_frames"].min()),
      float(index_clean["orig_num_pose_frames"].mean()),
      int(index_clean["orig_num_pose_frames"].max()))

nan_ratio_pose_xyz = float(np.isnan(X_pose[..., :3]).sum() / X_pose[..., :3].size)
nan_ratio_pose_velbone = float(np.isnan(X_pose[..., 4:10]).sum() / X_pose[..., 4:10].size)
nan_ratio_angles = float(np.isnan(X_angles).sum() / X_angles.size)

print("\nNaN ratio over X_pose xyz:", nan_ratio_pose_xyz)
print("NaN ratio over X_pose vel+bone:", nan_ratio_pose_velbone)
print("NaN ratio over X_angles:", nan_ratio_angles)

Label counts:
correctness
0    504
1    568
Name: count, dtype: int64

Exercise counts:
exercise_id
1    178
2    208
3    107
4    210
5    174
6    195
Name: count, dtype: int64

Unique persons: 10
Min / mean / max original frames: 15 119.81623134328358 585

NaN ratio over X_pose xyz: 0.0
NaN ratio over X_pose vel+bone: 0.0
NaN ratio over X_angles: 0.0


## 9) Inspect one processed sample

In [11]:
# 9) Inspect one processed sample
i = 0

print("Metadata row:")
display(index_clean.iloc[[i]])

print("Processed pose tensor shape:", X_pose[i].shape)
print("Processed angle tensor shape:", X_angles[i].shape)

print("\nFirst frame, first 3 landmarks from X_pose:")
print(X_pose[i, 0, :3, :])

print("\nFirst frame angle features:")
print(X_angles[i, 0])

Metadata row:


,video_id,repetition_number,exercise_id,person_id,first_frame,last_frame,cam17_orientation,mocap_erroneous,exercise_subtype,lights_on,extra_person_in_cam17,extra_person_in_cam18,correctness,video_path_abs,pose_path,num_pose_frames,pose_shape,orig_num_pose_frames,fixed_num_pose_frames
0,PM_000,1,1,1,180,377,front,0,right arm,0,3,0,1,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\d...,C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\c...,198,"(198, 33, 4)",198,32


Processed pose tensor shape: (32, 33, 10)
Processed angle tensor shape: (32, 8)

First frame, first 3 landmarks from X_pose:
[[-0.01990113 -1.3990937  -1.4413084   0.99999666  0.          0.
   0.          0.          0.          0.        ]
 [-0.00612071 -1.4550065  -1.3734392   0.99999297  0.          0.
   0.          0.          0.          0.        ]
 [ 0.00947364 -1.4521983  -1.3745269   0.99999106  0.          0.
   0.          0.          0.          0.        ]]

First frame angle features:
[2.0085783  2.2113478  2.3926904  2.407234   0.16546668 0.3173741
 2.6449049  2.5175552 ]
